# All cases clinical + longitudinal CNV/TFx extraction

This notebook uses the same approach as the 20-case extraction, but is configured for all cases present in the CNV matrix.

**Important:** if `cnv_matrix_file` still points to `Selected_representative...`, output will still be limited to that selected matrix. For true all-case extraction, change `cnv_matrix_file` and `cnv_annotation_file` to all-cases files.

Rules retained:
- Start from the CNV matrix so CNV-only cases are retained.
- `Active_treatment_at_draw` uses `treatment_start_day <= blood_draw_day <= treatment_end_day`.
- Missing treatment end day is not treated as ongoing.
- Treatment context before/after each draw is included.


In [1]:
# ============================================================
# All cases clinical + longitudinal CNV/TFx extraction
# ============================================================
#
# Purpose:
#   Generate All_cases_clinical_CNV_extraction.xlsx
#   and matching TSV files for all cases present in the CNV matrix.
#
# Important:
#   This script extracts ALL cases present in cnv_matrix_file.
#   If cnv_matrix_file still points to a Selected_representative...
#   matrix, the output will still be limited to those representative cases.
#   For true all-case extraction, point cnv_matrix_file and
#   cnv_annotation_file to the all-cases CNV matrix/annotation files.
#
# Important design:
#   1. Start from the CNV matrix, so CNV-only/unmapped cases are retained.
#   2. Map Patient_Identifier to clinical patient_id using clinical Barcode.
#   3. Strict Active_treatment_at_draw definition:
#        treatment_start_day <= blood_draw_day <= treatment_end_day
#      Both start and end days must be available.
#   4. Missing treatment end day is NOT assumed ongoing.
#   5. Active_treatment_at_draw alone is not enough, so the Blood_draws
#      table includes treatment context before/after each draw.
# ============================================================

import re
import math
import zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd


# ============================================================
# 1. User settings / paths
# ============================================================

# ---- Option A: CRC/HPC paths ----
BASE_DIR = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA")

clinical_file = BASE_DIR / "Clinical_data/AUP_ClinicalData_20260609.NO.DATES.bloodID.UseSet.txt"

# ------------------------------------------------------------------
# CNV input files
#
# For true all-case extraction, these should point to the all-cases
# longitudinal CNV matrix and matching sample annotation.
#
# If you keep the Selected_representative... files here, the script
# will extract all cases present in that selected matrix only.
# ------------------------------------------------------------------

cnv_matrix_file = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/CNVKit/All_case_CNV_extraction/All_representative_patients_longitudinal_gene_CNV_state_matrix.tsv")

cnv_annotation_file = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/CNVKit/All_case_CNV_extraction/All_representative_patients_longitudinal_CNV_sample_annotation.tsv")

cnr_manifest_file = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/CNVKit/All_case_CNV_extraction/All_representative_patients_longitudinal_CNR_manifest.tsv")
out_dir = BASE_DIR / "Clinical_data/All_cases_clinical_CNV_extraction_TSVs"

out_dir.mkdir(parents=True, exist_ok=True)

out_excel = out_dir / "All_cases_clinical_CNV_extraction.xlsx"
out_zip = out_dir / "All_cases_clinical_CNV_extraction_TSVs.zip"

# NULL/equivalent in Python:
# None = use all representative cases present in the CNV matrix.
candidate_cases = None

key_amp_genes = {
    "ERBB2", "FGFR1", "MYC", "CCND1", "MCL1",
    "PIK3CA", "ESR1", "CDH1", "TP53", "AKT1",
    "MAP3K1", "FOXA1"
}

treatment_categories = {
    "Met_Chemotherapy",
    "Met_Hormonal_Therapy",
    "Met_Targeted_Therapy",
    "Met_Immunotherapy",
    "Met_Other_Therapy"
}


# ============================================================
# 2. Helper functions
# ============================================================

def clean_blank(x):
    """Convert blank-like values to pandas NA, but keep clinical 'Unknown' values."""
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()

    if s == "" or s.lower() in {"na", "nan", "none", "null"}:
        return pd.NA

    return s


def clean_patient_id(x):
    """
    Clean clinical patient_id.

    Examples:
      1      -> "1"
      1.0    -> "1"
      "1.0"  -> "1"
      588-21 -> "588-21"
    """
    x = clean_blank(x)
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()

    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass

    return s


def extract_aup_identifier(x):
    """
    Extract the 4-character AURORA patient identifier.

    Examples:
      AUP-AQ0Z-TTP2-B -> AQ0Z
      AUP-AQ0Z-TTM1-A -> AQ0Z
      AUP-AMCZ-TTP1-B -> AMCZ
      AMCZ-NB1        -> AMCZ
      AQ30-NB1        -> AQ30
      AO01-NB1        -> AO01
      AM01-NB1        -> AM01
    """
    x = clean_blank(x)
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().upper()

    # Pattern 1: AUP-XXXX-...
    m = re.search(r"\bAUP-([A-Z0-9]{4})(?:-|$)", s)
    if m:
        return m.group(1)

    # Pattern 2: XXXX-NB1 / XXXX-NB2 etc.
    m = re.search(r"\b([A-Z0-9]{4})-NB\d+\b", s)
    if m:
        return m.group(1)

    # Fallback: first 4-character block before dash or end
    m = re.search(r"\b([A-Z0-9]{4})(?:-|$)", s)
    if m:
        return m.group(1)

    return pd.NA


def extract_draw_num(x):
    x = clean_blank(x)
    if pd.isna(x):
        return np.nan

    m = re.search(r"NB(\d+)", str(x).upper())
    if not m:
        return np.nan

    return float(m.group(1))


def to_numeric_or_na(x):
    return pd.to_numeric(x, errors="coerce")


def first_non_missing(series):
    vals = [clean_blank(v) for v in series]
    vals = [v for v in vals if not pd.isna(v)]
    return vals[0] if vals else pd.NA


def paste_unique(series, sep="; "):
    vals = []
    seen = set()
    for v in series:
        v = clean_blank(v)
        if pd.isna(v):
            continue
        v = str(v)
        if v not in seen:
            vals.append(v)
            seen.add(v)
    return sep.join(vals) if vals else pd.NA


def interval_label(start, end):
    if pd.isna(start) and pd.isna(end):
        return "day unknown"
    if not pd.isna(start) and pd.isna(end):
        return f"day {start:g}-unknown end"
    if pd.isna(start) and not pd.isna(end):
        return f"until day {end:g}"
    return f"day {start:g}-{end:g}"


def treatment_label(row):
    line = clean_blank(row.get("Met_Therapy_Line"))
    regimen = clean_blank(row.get("Regimen"))
    category = clean_blank(row.get("Category"))

    line = "Unknown line" if pd.isna(line) else line
    regimen = "Unknown regimen" if pd.isna(regimen) else regimen
    category = "Unknown category" if pd.isna(category) else category

    return f"{line}: {regimen} ({category})"


def make_tx_summary_label(row):
    out = f"{row['treatment_label']} [{row['treatment_interval']}]"

    if not pd.isna(clean_blank(row.get("treatment_response"))):
        out += f", response={row.get('treatment_response')}"

    if not pd.isna(clean_blank(row.get("progression"))):
        out += f", progression={row.get('progression')}"

    if not pd.isna(clean_blank(row.get("reason_tx_ended"))):
        out += f", reason ended={row.get('reason_tx_ended')}"

    return out


def safe_minmax_norm(x):
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    if x.notna().sum() == 0:
        return pd.Series(np.zeros(len(x)), index=x.index)

    mn = x.min(skipna=True)
    mx = x.max(skipna=True)

    if mn == mx:
        return pd.Series(np.zeros(len(x)), index=x.index)

    return (x.fillna(mn) - mn) / (mx - mn)


def require_file(path):
    if not Path(path).exists():
        raise FileNotFoundError(f"File not found: {path}")
    return Path(path)


# ============================================================
# 3. Read input files
# ============================================================

clinical_file = require_file(clinical_file)
cnv_matrix_file = require_file(cnv_matrix_file)
cnv_annotation_file = require_file(cnv_annotation_file)

# CNR manifest is useful for traceability but not required for the extraction.
if not cnr_manifest_file.exists():
    print(f"WARNING: CNR manifest not found, continuing without it: {cnr_manifest_file}")

clinical = pd.read_csv(
    clinical_file,
    sep="\t",
    dtype=str,
    keep_default_na=False,
    na_values=[],
    encoding="latin1"
)

cnv_matrix = pd.read_csv(
    cnv_matrix_file,
    sep="\t",
    dtype=str,
    keep_default_na=False,
    na_values=[]
)

cnv_annot = pd.read_csv(
    cnv_annotation_file,
    sep="\t",
    dtype=str,
    keep_default_na=False,
    na_values=[]
)

# Clean core clinical columns
clinical["patient_id"] = clinical["patient_id"].apply(clean_patient_id)
clinical["Site"] = clinical["Site"].apply(clean_blank)
clinical["Category"] = clinical["Category"].apply(clean_blank)
clinical["Barcode"] = clinical["Barcode"].apply(clean_blank)
clinical["Clinical_ID"] = clinical["Clinical_ID"].apply(clean_blank)
clinical["Clinical_ID_2"] = clinical["Clinical_ID_2"].apply(clean_blank)
clinical["Days_From_Primary_Diagnosis_num"] = pd.to_numeric(
    clinical["Days_From_Primary_Diagnosis"], errors="coerce"
)
clinical["EndTrt_Days_From_Primary_Diagnosis_num"] = pd.to_numeric(
    clinical["EndTrt_Days_From_Primary_Diagnosis"], errors="coerce"
)

print("Clinical rows:", clinical.shape)
print("CNV matrix:", cnv_matrix.shape)
print("CNV annotation:", cnv_annot.shape)


# ============================================================
# 4. CNV sample and patient identifiers
# ============================================================

gene_col = "gene" if "gene" in cnv_matrix.columns else cnv_matrix.columns[0]
cnv_matrix = cnv_matrix.rename(columns={gene_col: "gene"})

cnv_sample_cols = [c for c in cnv_matrix.columns if c != "gene"]

sample_to_patient = {
    sample: extract_aup_identifier(sample)
    for sample in cnv_sample_cols
}

sample_to_draw = {
    sample: extract_draw_num(sample)
    for sample in cnv_sample_cols
}

cnv_patients_all = sorted(
    {
        pid for pid in sample_to_patient.values()
        if not pd.isna(pid)
    }
)

if candidate_cases is not None:
    cnv_patients_all = [x for x in cnv_patients_all if x in set(candidate_cases)]

print("Cases present in CNV matrix:", len(cnv_patients_all))
print(cnv_patients_all)

if any("Selected_representative" in str(p) for p in [cnv_matrix_file, cnv_annotation_file]):
    print("\nWARNING:")
    print("  Your CNV input path contains 'Selected_representative'.")
    print("  This script will extract all cases present in that selected matrix.")
    print("  For true all-case extraction, change cnv_matrix_file and")
    print("  cnv_annotation_file to all-cases matrix/annotation files.\n")


# ============================================================
# 5. Build clinical Patient_Identifier → patient_id mapping
# ============================================================

clinical_xwalk = clinical.copy()
clinical_xwalk["Patient_Identifier"] = clinical_xwalk["Barcode"].apply(extract_aup_identifier)

clinical_xwalk = clinical_xwalk[
    clinical_xwalk["Patient_Identifier"].notna()
    & clinical_xwalk["patient_id"].notna()
].copy()

map_rows = []

for pid4 in cnv_patients_all:
    sub = clinical_xwalk[clinical_xwalk["Patient_Identifier"] == pid4].copy()

    patient_ids = sorted(sub["patient_id"].dropna().unique().tolist())
    sites = sub["Site"].dropna().unique().tolist()

    if len(patient_ids) == 1:
        patient_id = patient_ids[0]
        mapping_status = "Mapped to clinical patient_id"
    elif len(patient_ids) > 1:
        patient_id = pd.NA
        mapping_status = "Conflicting clinical patient_id mapping"
    else:
        patient_id = pd.NA
        mapping_status = "No clinical Barcode match"

    map_rows.append({
        "Patient_Identifier": pid4,
        "patient_id": patient_id,
        "Site": first_non_missing(sites),
        "mapping_status": mapping_status,
        "patient_ids_all": "; ".join(patient_ids) if patient_ids else pd.NA,
        "n_clinical_barcode_rows": int(sub.shape[0]),
        "example_barcodes": paste_unique(sub["Barcode"].head(10))
    })

candidate_patient_map = pd.DataFrame(map_rows)

print(candidate_patient_map)
print("\nMapping summary:")
print(candidate_patient_map["mapping_status"].value_counts(dropna=False))


# ============================================================
# 6. CNV wide matrix setup
# ============================================================

state_mat = cnv_matrix.set_index("gene")[cnv_sample_cols].copy()

state_mat = (
    state_mat
    .fillna("NEUT")
    .astype(str)
    .apply(lambda col: col.str.strip().str.upper())
)

state_mat = state_mat.replace({
    "": "NEUT",
    "NA": "NEUT",
    "NAN": "NEUT",
    "NONE": "NEUT",
    "NULL": "NEUT"
})

altered_mat = state_mat.ne("NEUT")
amp_mat = state_mat.isin(["AMP", "HLAMP"])

print("CNV wide matrix:")
print("  genes:", state_mat.shape[0])
print("  samples:", state_mat.shape[1])
print("  total gene-sample cells:", state_mat.shape[0] * state_mat.shape[1])


# ============================================================
# 7. CNV annotation cleanup
# ============================================================

sample_col_candidates = ["sample_id", "Sample_ID", "sample", "Barcode", "Collaborator_sample_id"]
sample_col = next((c for c in sample_col_candidates if c in cnv_annot.columns), cnv_annot.columns[0])

cnv_annot["sample_id"] = cnv_annot[sample_col].apply(clean_blank)

if "Patient_Identifier" in cnv_annot.columns:
    cnv_annot["Patient_Identifier"] = cnv_annot["Patient_Identifier"].apply(clean_blank)
else:
    cnv_annot["Patient_Identifier"] = cnv_annot["sample_id"].apply(extract_aup_identifier)

# TFx column handling
if "Tumor_Fraction_percent" in cnv_annot.columns:
    cnv_annot["Tumor_Fraction_percent"] = pd.to_numeric(cnv_annot["Tumor_Fraction_percent"], errors="coerce")
elif "TFx_percent" in cnv_annot.columns:
    cnv_annot["Tumor_Fraction_percent"] = pd.to_numeric(cnv_annot["TFx_percent"], errors="coerce")
elif "Tumor_Fraction" in cnv_annot.columns:
    tmp = pd.to_numeric(cnv_annot["Tumor_Fraction"], errors="coerce")
    cnv_annot["Tumor_Fraction_percent"] = np.where(tmp <= 1, tmp * 100, tmp)
else:
    cnv_annot["Tumor_Fraction_percent"] = np.nan

if "Histology_group" not in cnv_annot.columns:
    cnv_annot["Histology_group"] = pd.NA

if "Molecular_subtype_clean" not in cnv_annot.columns:
    cnv_annot["Molecular_subtype_clean"] = pd.NA

cnv_annot_small = cnv_annot[
    ["sample_id", "Patient_Identifier", "Tumor_Fraction_percent", "Histology_group", "Molecular_subtype_clean"]
].drop_duplicates()


# ============================================================
# 8. CNV sample summary from wide matrix
# ============================================================

sample_rows = []

for sample in cnv_sample_cols:
    pid4 = sample_to_patient.get(sample)
    draw_num = sample_to_draw.get(sample)

    sample_states = state_mat[sample]
    altered = sample_states[sample_states != "NEUT"]
    amp = sample_states[sample_states.isin(["AMP", "HLAMP"])]

    sample_rows.append({
        "Patient_Identifier": pid4,
        "sample_id": sample,
        "draw_num": draw_num,
        "CNV_burden_non_NEUT_genes": int(altered.shape[0]),
        "Amplified_genes_HLAMP_or_AMP": paste_unique(
            amp.reset_index().apply(
                lambda r: f"{r['gene']}:{r[sample]}",
                axis=1
            )
        ),
        # For all-case analysis, this can be very large.
        # Keep it if useful, otherwise comment it out.
        "Altered_genes_all_non_NEUT": paste_unique(
            altered.reset_index().apply(
                lambda r: f"{r['gene']}:{r[sample]}",
                axis=1
            )
        )
    })

cnv_sample_summary = pd.DataFrame(sample_rows)

cnv_sample_summary = cnv_sample_summary[
    cnv_sample_summary["Patient_Identifier"].isin(cnv_patients_all)
].copy()

cnv_sample_summary = cnv_sample_summary.merge(
    cnv_annot_small,
    on=["sample_id", "Patient_Identifier"],
    how="left"
)

candidate_cases_CNV_sample_summary = (
    cnv_sample_summary
    .merge(
        candidate_patient_map[["Patient_Identifier", "patient_id", "Site", "mapping_status"]],
        on="Patient_Identifier",
        how="left"
    )
)

candidate_cases_CNV_sample_summary = candidate_cases_CNV_sample_summary[
    [
        "Patient_Identifier", "patient_id", "Site", "mapping_status",
        "sample_id", "draw_num", "Tumor_Fraction_percent",
        "Histology_group", "Molecular_subtype_clean",
        "CNV_burden_non_NEUT_genes",
        "Amplified_genes_HLAMP_or_AMP",
        "Altered_genes_all_non_NEUT"
    ]
].sort_values(["Patient_Identifier", "draw_num", "sample_id"])

# ============================================================
# 9. CNV key-gene summary only
# ============================================================

key_gene_rows = []

for pid4 in cnv_patients_all:
    patient_samples = [
        s for s in cnv_sample_cols
        if sample_to_patient.get(s) == pid4
    ]

    if len(patient_samples) == 0:
        continue

    sub = state_mat[patient_samples]

    for gene in key_amp_genes:
        if gene not in sub.index:
            continue

        states = sub.loc[gene].dropna().astype(str)
        states_observed = sorted(states.unique().tolist())
        non_neut_states = [s for s in states_observed if s != "NEUT"]

        altered_samples = states[states != "NEUT"]
        amp_samples = states[states.isin(["AMP", "HLAMP"])]

        key_gene_rows.append({
            "Patient_Identifier": pid4,
            "gene": gene,
            "n_samples": len(patient_samples),
            "states_observed": ",".join(states_observed),
            "non_NEUT_states_observed": ",".join(non_neut_states) if non_neut_states else pd.NA,
            "n_samples_altered": int(altered_samples.shape[0]),
            "n_samples_amp_or_hlamp": int(amp_samples.shape[0]),
            "variable_gene": len(states_observed) > 1,
            "sample_state_series": paste_unique(
                [
                    f"{sample}:{state}"
                    for sample, state in altered_samples.items()
                ]
            )
        })

candidate_cases_CNV_gene_summary_key_genes = pd.DataFrame(key_gene_rows)

candidate_cases_CNV_gene_summary_key_genes = (
    candidate_cases_CNV_gene_summary_key_genes
    .merge(
        candidate_patient_map[["Patient_Identifier", "patient_id", "Site", "mapping_status"]],
        on="Patient_Identifier",
        how="left"
    )
)

# ============================================================
# 10. CNV rank features
# ============================================================

rank_rows = []

for pid4 in cnv_patients_all:
    patient_samples = [
        s for s in cnv_sample_cols
        if sample_to_patient.get(s) == pid4
    ]

    patient_samples = sorted(
        patient_samples,
        key=lambda s: (
            sample_to_draw.get(s) if not pd.isna(sample_to_draw.get(s)) else 9999,
            s
        )
    )

    ss = cnv_sample_summary[cnv_sample_summary["Patient_Identifier"] == pid4].copy()

    if len(patient_samples) == 0:
        continue

    sub = state_mat[patient_samples]

    tfx = pd.to_numeric(ss["Tumor_Fraction_percent"], errors="coerce").dropna()
    burden = pd.to_numeric(ss["CNV_burden_non_NEUT_genes"], errors="coerce").dropna()

    n_variable_genes = int((sub.nunique(axis=1) > 1).sum())

    if sub.shape[1] >= 2:
        total_gene_state_changes = int(
            sub.ne(sub.shift(axis=1)).iloc[:, 1:].sum(axis=1).sum()
        )
    else:
        total_gene_state_changes = 0

    key_genes_present = [g for g in key_amp_genes if g in sub.index]

    if len(key_genes_present) > 0:
        key_sub = sub.loc[key_genes_present]
        key_amp_calls = int(key_sub.isin(["AMP", "HLAMP"]).sum().sum())

        notable_list = []
        for gene in key_genes_present:
            states = sorted(set(key_sub.loc[gene].astype(str)) - {"NEUT"})
            if len(states) > 0:
                notable_list.append(f"{gene}({','.join(states)})")
    else:
        key_amp_calls = 0
        notable_list = []

    rank_rows.append({
        "Patient_Identifier": pid4,
        "n_draws": int(len(patient_samples)),
        "max_TFx": tfx.max() if len(tfx) else np.nan,
        "min_TFx": tfx.min() if len(tfx) else np.nan,
        "mean_TFx": tfx.mean() if len(tfx) else np.nan,
        "range_TFx": (tfx.max() - tfx.min()) if len(tfx) >= 2 else 0,
        "mean_CNV_burden": burden.mean() if len(burden) else np.nan,
        "max_CNV_burden": burden.max() if len(burden) else np.nan,
        "min_CNV_burden": burden.min() if len(burden) else np.nan,
        "range_CNV_burden": (burden.max() - burden.min()) if len(burden) else 0,
        "n_variable_genes": n_variable_genes,
        "total_gene_state_changes": total_gene_state_changes,
        "key_amp_calls": key_amp_calls,
        "notable_genes_states": paste_unique(notable_list),
        "histology_from_ranked_or_annotation": first_non_missing(ss["Histology_group"]),
        "subtype_from_ranked_or_annotation": first_non_missing(ss["Molecular_subtype_clean"]),
    })

rank_features_all = pd.DataFrame(rank_rows)

rank_features_all = rank_features_all.merge(
    candidate_patient_map[["Patient_Identifier", "patient_id", "Site", "mapping_status"]],
    on="Patient_Identifier",
    how="left"
)

for feat in [
    "n_draws", "max_TFx", "range_TFx", "mean_CNV_burden",
    "range_CNV_burden", "n_variable_genes",
    "total_gene_state_changes", "key_amp_calls"
]:
    rank_features_all[f"norm_{feat}"] = safe_minmax_norm(rank_features_all[feat])

rank_features_all["CNV_rank_score"] = (
    1.2 * rank_features_all["norm_n_draws"] +
    1.2 * rank_features_all["norm_max_TFx"] +
    1.0 * rank_features_all["norm_range_TFx"] +
    1.2 * rank_features_all["norm_mean_CNV_burden"] +
    1.5 * rank_features_all["norm_range_CNV_burden"] +
    1.5 * rank_features_all["norm_n_variable_genes"] +
    1.0 * rank_features_all["norm_total_gene_state_changes"] +
    1.3 * rank_features_all["norm_key_amp_calls"]
).round(2)

rank_features_all = rank_features_all.sort_values(
    "CNV_rank_score",
    ascending=False
)
# ============================================================
# 11. Clinical rows for mapped candidate cases
# ============================================================

mapped_patient_ids = (
    candidate_patient_map
    .loc[candidate_patient_map["mapping_status"] == "Mapped to clinical patient_id", "patient_id"]
    .dropna()
    .astype(str)
    .tolist()
)

patient_id_to_pid4 = dict(
    zip(
        candidate_patient_map["patient_id"].astype(str),
        candidate_patient_map["Patient_Identifier"]
    )
)

candidate_cases_all_clinical_rows = clinical[
    clinical["patient_id"].astype(str).isin(mapped_patient_ids)
].copy()

candidate_cases_all_clinical_rows["Patient_Identifier"] = (
    candidate_cases_all_clinical_rows["patient_id"].astype(str).map(patient_id_to_pid4)
)

# Put Patient_Identifier first
front = ["Patient_Identifier"]
candidate_cases_all_clinical_rows = candidate_cases_all_clinical_rows[
    front + [c for c in candidate_cases_all_clinical_rows.columns if c not in front]
].copy()

# Drop helper numeric columns from the exported clinical-row sheet
candidate_cases_all_clinical_rows_export = candidate_cases_all_clinical_rows.drop(
    columns=["Days_From_Primary_Diagnosis_num", "EndTrt_Days_From_Primary_Diagnosis_num"],
    errors="ignore"
).sort_values(
    ["Patient_Identifier", "Days_From_Primary_Diagnosis", "Category"],
    na_position="last"
)


# ============================================================
# 12. Treatment timeline
# ============================================================

tx = candidate_cases_all_clinical_rows[
    candidate_cases_all_clinical_rows["Category"].isin(treatment_categories)
].copy()

tx["treatment_start_day"] = tx["Days_From_Primary_Diagnosis_num"]
tx["treatment_end_day"] = tx["EndTrt_Days_From_Primary_Diagnosis_num"]
tx["treatment_label"] = tx.apply(treatment_label, axis=1)
tx["treatment_interval"] = tx.apply(
    lambda r: interval_label(r["treatment_start_day"], r["treatment_end_day"]),
    axis=1
)

candidate_cases_treatment_timeline = tx.rename(columns={
    "Category": "treatment_category",
    "Met_Therapy_Line": "treatment_line",
    "Regimen": "regimen",
    "Treatment_Response": "treatment_response",
    "Progression": "progression",
    "Reason_Tx_Ended": "reason_tx_ended",
})[
    [
        "Patient_Identifier", "patient_id", "Site",
        "treatment_category", "treatment_start_day", "treatment_end_day",
        "treatment_line", "regimen",
        "treatment_response", "progression", "reason_tx_ended",
        "treatment_label", "treatment_interval"
    ]
].copy()

candidate_cases_treatment_timeline = candidate_cases_treatment_timeline.sort_values(
    ["Patient_Identifier", "treatment_start_day", "treatment_category", "regimen"],
    na_position="last"
)


# ============================================================
# 13. Diagnosis / receptor / tissue rows
# ============================================================

diagnosis_categories = {
    "First_Primary", "Second_Primary", "Primary_Diagnosis",
    "Met_Diagnosis", "Metastatic_Diagnosis", "Diagnosis"
}

candidate_cases_diagnosis_receptor_tissue = candidate_cases_all_clinical_rows[
    candidate_cases_all_clinical_rows["Category"].isin(diagnosis_categories)
    | candidate_cases_all_clinical_rows["Histologic.type"].apply(lambda x: not pd.isna(clean_blank(x)))
    | candidate_cases_all_clinical_rows["ER_Status"].apply(lambda x: not pd.isna(clean_blank(x)))
    | candidate_cases_all_clinical_rows["PR_Status"].apply(lambda x: not pd.isna(clean_blank(x)))
    | candidate_cases_all_clinical_rows["HER2_Status"].apply(lambda x: not pd.isna(clean_blank(x)))
    | candidate_cases_all_clinical_rows["Tissue_Site"].apply(lambda x: not pd.isna(clean_blank(x)))
].copy()

diagnosis_cols = [
    "Patient_Identifier", "patient_id", "Site", "Category",
    "Days_From_Primary_Diagnosis", "Barcode", "Clinical_ID", "Clinical_ID_2",
    "Tissue_Site", "bx", "bx.details", "Histologic.type",
    "ER_Status", "ER_pct", "PR_Status", "PR_pct",
    "HER2_Status", "HER2_IHC", "HER2_Fish", "HER2_CN",
    "Preservation", "Comment"
]

candidate_cases_diagnosis_receptor_tissue = candidate_cases_diagnosis_receptor_tissue[
    [c for c in diagnosis_cols if c in candidate_cases_diagnosis_receptor_tissue.columns]
].sort_values(
    ["Patient_Identifier", "Days_From_Primary_Diagnosis", "Category"],
    na_position="last"
)


# ============================================================
# 14. Blood draws with treatment context
# ============================================================

clinical_blood_draws = candidate_cases_all_clinical_rows[
    candidate_cases_all_clinical_rows["Category"] == "Blood_Draw"
].copy()

blood_lookup = {
    (row["Patient_Identifier"], row["Barcode"]): row
    for _, row in clinical_blood_draws.iterrows()
    if not pd.isna(clean_blank(row.get("Barcode")))
}

tx_by_patient_id = {
    str(pid): sub.copy()
    for pid, sub in candidate_cases_treatment_timeline.groupby("patient_id", dropna=False)
}


def get_treatment_context(patient_id, draw_day):
    """
    Treatment context for a blood draw.

    Strict active-treatment definition:
        treatment_start_day <= draw_day <= treatment_end_day

    Both treatment_start_day and treatment_end_day must be present.
    Missing treatment end day is NOT assumed ongoing.
    """
    empty = {
        "Active_treatment_at_draw": pd.NA,
        "Treatment_context_at_draw": (
            "No clinical blood draw date or patient_id mapping"
            if pd.isna(patient_id) or pd.isna(draw_day)
            else "No matched treatment context"
        ),

        "Most_recent_treatment_before_draw": pd.NA,
        "Most_recent_treatment_start_day": np.nan,
        "Most_recent_treatment_end_day": np.nan,
        "Most_recent_treatment_response_recorded": pd.NA,
        "Most_recent_treatment_progression_recorded": pd.NA,
        "Most_recent_treatment_reason_ended_recorded": pd.NA,

        "Most_recent_completed_treatment_before_draw": pd.NA,
        "Most_recent_completed_treatment_end_day": np.nan,
        "Days_since_most_recent_completed_treatment_end": np.nan,
        "Most_recent_completed_treatment_response_recorded": pd.NA,
        "Most_recent_completed_treatment_progression_recorded": pd.NA,
        "Most_recent_completed_treatment_reason_ended_recorded": pd.NA,

        "Next_treatment_after_draw": pd.NA,
        "Next_treatment_start_day": np.nan,
        "Days_until_next_treatment_start": np.nan,
    }

    if pd.isna(patient_id) or pd.isna(draw_day):
        return empty

    tx_patient = tx_by_patient_id.get(str(patient_id))

    if tx_patient is None or tx_patient.empty:
        return empty

    # Strict active overlap:
    active = tx_patient[
        tx_patient["treatment_start_day"].notna()
        & tx_patient["treatment_end_day"].notna()
        & (tx_patient["treatment_start_day"] <= draw_day)
        & (tx_patient["treatment_end_day"] >= draw_day)
    ].copy()

    recent_started = tx_patient[
        tx_patient["treatment_start_day"].notna()
        & (tx_patient["treatment_start_day"] <= draw_day)
    ].sort_values(
        ["treatment_start_day", "treatment_end_day"],
        ascending=[False, False],
        na_position="last"
    )

    recent_completed = tx_patient[
        tx_patient["treatment_end_day"].notna()
        & (tx_patient["treatment_end_day"] < draw_day)
    ].sort_values(
        ["treatment_end_day", "treatment_start_day"],
        ascending=[False, False],
        na_position="last"
    )

    next_tx = tx_patient[
        tx_patient["treatment_start_day"].notna()
        & (tx_patient["treatment_start_day"] > draw_day)
    ].sort_values(
        ["treatment_start_day", "treatment_end_day"],
        ascending=[True, True],
        na_position="last"
    )

    active_label = paste_unique(active["treatment_label"]) if not active.empty else pd.NA

    recent_label = pd.NA
    recent_start = np.nan
    recent_end = np.nan
    recent_response = pd.NA
    recent_progression = pd.NA
    recent_reason = pd.NA

    if not recent_started.empty:
        recent_start = recent_started.iloc[0]["treatment_start_day"]
        recent_end = recent_started.iloc[0]["treatment_end_day"]
        same_start = recent_started[recent_started["treatment_start_day"] == recent_start]

        recent_label = paste_unique(same_start["treatment_label"])
        recent_response = paste_unique(same_start["treatment_response"])
        recent_progression = paste_unique(same_start["progression"])
        recent_reason = paste_unique(same_start["reason_tx_ended"])

    completed_label = pd.NA
    completed_end = np.nan
    days_since_completed_end = np.nan
    completed_response = pd.NA
    completed_progression = pd.NA
    completed_reason = pd.NA

    if not recent_completed.empty:
        completed_end = recent_completed.iloc[0]["treatment_end_day"]
        days_since_completed_end = draw_day - completed_end

        same_end = recent_completed[recent_completed["treatment_end_day"] == completed_end]

        completed_label = paste_unique(same_end["treatment_label"])
        completed_response = paste_unique(same_end["treatment_response"])
        completed_progression = paste_unique(same_end["progression"])
        completed_reason = paste_unique(same_end["reason_tx_ended"])

    next_label = pd.NA
    next_start = np.nan
    days_until_next_start = np.nan

    if not next_tx.empty:
        next_start = next_tx.iloc[0]["treatment_start_day"]
        days_until_next_start = next_start - draw_day
        same_next = next_tx[next_tx["treatment_start_day"] == next_start]
        next_label = paste_unique(same_next["treatment_label"])

    if not pd.isna(active_label):
        context = "On-treatment"
    elif not pd.isna(completed_label) and not pd.isna(next_label):
        context = "Between-treatment-lines"
    elif not pd.isna(completed_label) and pd.isna(next_label):
        context = "Post-treatment"
    elif pd.isna(completed_label) and not pd.isna(next_label):
        context = "Pre-treatment"
    else:
        context = "No matched treatment context"

    return {
        "Active_treatment_at_draw": active_label,
        "Treatment_context_at_draw": context,

        "Most_recent_treatment_before_draw": recent_label,
        "Most_recent_treatment_start_day": recent_start,
        "Most_recent_treatment_end_day": recent_end,
        "Most_recent_treatment_response_recorded": recent_response,
        "Most_recent_treatment_progression_recorded": recent_progression,
        "Most_recent_treatment_reason_ended_recorded": recent_reason,

        "Most_recent_completed_treatment_before_draw": completed_label,
        "Most_recent_completed_treatment_end_day": completed_end,
        "Days_since_most_recent_completed_treatment_end": days_since_completed_end,
        "Most_recent_completed_treatment_response_recorded": completed_response,
        "Most_recent_completed_treatment_progression_recorded": completed_progression,
        "Most_recent_completed_treatment_reason_ended_recorded": completed_reason,

        "Next_treatment_after_draw": next_label,
        "Next_treatment_start_day": next_start,
        "Days_until_next_treatment_start": days_until_next_start,
    }


blood_rows = []

for _, srow in cnv_sample_summary.iterrows():
    pid4 = srow["Patient_Identifier"]
    sample = srow["sample_id"]

    md = candidate_patient_map[candidate_patient_map["Patient_Identifier"] == pid4].iloc[0]

    cinfo = blood_lookup.get((pid4, sample), None)

    patient_id = md["patient_id"]
    site = md["Site"]
    draw_day = np.nan
    clinical_id = pd.NA
    clinical_id_2 = pd.NA

    if cinfo is not None:
        if pd.isna(patient_id):
            patient_id = cinfo.get("patient_id")
        if pd.isna(site):
            site = cinfo.get("Site")
        draw_day = cinfo.get("Days_From_Primary_Diagnosis_num")
        clinical_id = cinfo.get("Clinical_ID")
        clinical_id_2 = cinfo.get("Clinical_ID_2")

    context = get_treatment_context(patient_id, draw_day)

    blood_rows.append({
        "Patient_Identifier": pid4,
        "patient_id": patient_id,
        "Site": site,
        "mapping_status": md["mapping_status"],
        "Blood_Draw_Days_From_Primary_Diagnosis": draw_day,
        "Barcode": sample,
        "Clinical_ID": clinical_id,
        "Clinical_ID_2": clinical_id_2,
        **context,
        "draw_num": srow["draw_num"],
        "Tumor_Fraction_percent_from_CNV_annotation": srow["Tumor_Fraction_percent"],
        "Histological_subtype_from_CNV_annotation": srow["Histology_group"],
        "Molecular_subtype_from_CNV_annotation": srow["Molecular_subtype_clean"],
        "CNV_burden_non_NEUT_genes": srow["CNV_burden_non_NEUT_genes"],
        "Amplified_genes_HLAMP_or_AMP": srow["Amplified_genes_HLAMP_or_AMP"],
        "Altered_genes_all_non_NEUT": srow["Altered_genes_all_non_NEUT"],
    })

candidate_cases_blood_draws_with_treatment_CNV_TFx = pd.DataFrame(blood_rows)

candidate_cases_blood_draws_with_treatment_CNV_TFx = (
    candidate_cases_blood_draws_with_treatment_CNV_TFx
    .sort_values(
        ["Patient_Identifier", "Blood_Draw_Days_From_Primary_Diagnosis", "draw_num", "Barcode"],
        na_position="last"
    )
    .copy()
)

# Add previous draw deltas
candidate_cases_blood_draws_with_treatment_CNV_TFx[
    "Previous_blood_draw_day"
] = candidate_cases_blood_draws_with_treatment_CNV_TFx.groupby("Patient_Identifier")[
    "Blood_Draw_Days_From_Primary_Diagnosis"
].shift(1)

candidate_cases_blood_draws_with_treatment_CNV_TFx[
    "Days_since_previous_blood_draw"
] = (
    candidate_cases_blood_draws_with_treatment_CNV_TFx["Blood_Draw_Days_From_Primary_Diagnosis"]
    - candidate_cases_blood_draws_with_treatment_CNV_TFx["Previous_blood_draw_day"]
)

candidate_cases_blood_draws_with_treatment_CNV_TFx[
    "Previous_TFx_percent"
] = candidate_cases_blood_draws_with_treatment_CNV_TFx.groupby("Patient_Identifier")[
    "Tumor_Fraction_percent_from_CNV_annotation"
].shift(1)

candidate_cases_blood_draws_with_treatment_CNV_TFx[
    "TFx_change_from_previous_draw"
] = (
    candidate_cases_blood_draws_with_treatment_CNV_TFx["Tumor_Fraction_percent_from_CNV_annotation"]
    - candidate_cases_blood_draws_with_treatment_CNV_TFx["Previous_TFx_percent"]
)

candidate_cases_blood_draws_with_treatment_CNV_TFx[
    "Previous_CNV_burden_non_NEUT_genes"
] = candidate_cases_blood_draws_with_treatment_CNV_TFx.groupby("Patient_Identifier")[
    "CNV_burden_non_NEUT_genes"
].shift(1)

candidate_cases_blood_draws_with_treatment_CNV_TFx[
    "CNV_burden_change_from_previous_draw"
] = (
    candidate_cases_blood_draws_with_treatment_CNV_TFx["CNV_burden_non_NEUT_genes"]
    - candidate_cases_blood_draws_with_treatment_CNV_TFx["Previous_CNV_burden_non_NEUT_genes"]
)


# ============================================================
# 15. Case summary
# ============================================================

clinical_counts = (
    candidate_cases_all_clinical_rows
    .groupby("Patient_Identifier", dropna=False)
    .size()
    .rename("n_all_clinical_rows")
    .reset_index()
)

blood_days_summary = (
    candidate_cases_blood_draws_with_treatment_CNV_TFx
    .groupby("Patient_Identifier", dropna=False)
    .apply(
        lambda df: pd.Series({
            "n_blood_draw_rows_in_clinical": int(df["Blood_Draw_Days_From_Primary_Diagnosis"].notna().sum()),
            "blood_draw_days": paste_unique(
                df.loc[
                    df["Blood_Draw_Days_From_Primary_Diagnosis"].notna()
                ].apply(
                    lambda r: f"{r['Barcode']}:{r['Blood_Draw_Days_From_Primary_Diagnosis']:g}",
                    axis=1
                )
            )
        })
    )
    .reset_index()
)

# Receptor summary
receptor_rows = []

for pid4, sub in candidate_cases_diagnosis_receptor_tissue.groupby("Patient_Identifier", dropna=False):
    labels = []

    for _, r in sub.iterrows():
        if (
            not pd.isna(clean_blank(r.get("Histologic.type")))
            or not pd.isna(clean_blank(r.get("ER_Status")))
            or not pd.isna(clean_blank(r.get("PR_Status")))
            or not pd.isna(clean_blank(r.get("HER2_Status")))
        ):
            tissue = clean_blank(r.get("Tissue_Site"))
            tissue_part = f" [{tissue}]" if not pd.isna(tissue) else ""

            labels.append(
                f"{r.get('Category')} day {r.get('Days_From_Primary_Diagnosis')}{tissue_part}: "
                f"Hist={r.get('Histologic.type') if not pd.isna(clean_blank(r.get('Histologic.type'))) else 'NA'}, "
                f"ER={r.get('ER_Status') if not pd.isna(clean_blank(r.get('ER_Status'))) else 'NA'}, "
                f"PR={r.get('PR_Status') if not pd.isna(clean_blank(r.get('PR_Status'))) else 'NA'}, "
                f"HER2={r.get('HER2_Status') if not pd.isna(clean_blank(r.get('HER2_Status'))) else 'NA'}"
            )

    receptor_rows.append({
        "Patient_Identifier": pid4,
        "receptor_summary_from_clinical": paste_unique(labels)
    })

receptor_summary = pd.DataFrame(receptor_rows)

# Metastatic summary
met_rows = []

for pid4, sub in candidate_cases_all_clinical_rows.groupby("Patient_Identifier", dropna=False):
    labels = []

    for _, r in sub[sub["Category"] == "Met_Diagnosis"].iterrows():
        label = f"day {r.get('Days_From_Primary_Diagnosis')}: {r.get('Tissue_Site') if not pd.isna(clean_blank(r.get('Tissue_Site'))) else 'site NA'}"

        if not pd.isna(clean_blank(r.get("bx"))):
            label += f", bx={r.get('bx')}"

        if not pd.isna(clean_blank(r.get("bx.details"))):
            label += f", {r.get('bx.details')}"

        labels.append(label)

    met_rows.append({
        "Patient_Identifier": pid4,
        "metastatic_sites_summary": paste_unique(labels)
    })

metastatic_summary = pd.DataFrame(met_rows)

# Treatment summary
tx_summary = (
    candidate_cases_treatment_timeline
    .assign(tx_summary_label=lambda df: df.apply(make_tx_summary_label, axis=1))
    .groupby("Patient_Identifier", dropna=False)["tx_summary_label"]
    .apply(paste_unique)
    .rename("treatment_summary")
    .reset_index()
)

# Survival summary
survival_rows = []

for pid4, sub in candidate_cases_all_clinical_rows.groupby("Patient_Identifier", dropna=False):
    labels = []

    for _, r in sub.iterrows():
        if r.get("Category") == "Overall_Survival_days":
            labels.append(
                f"{r.get('BCR_Status') if not pd.isna(clean_blank(r.get('BCR_Status'))) else 'Survival status unknown'} "
                f"at {r.get('Days_From_Primary_Diagnosis')} days"
            )
        elif r.get("Category") == "date_of_death":
            labels.append(
                f"Date of death row at {r.get('Days_From_Primary_Diagnosis')} days"
            )

    survival_rows.append({
        "Patient_Identifier": pid4,
        "survival_summary": paste_unique(labels)
    })

survival_summary = pd.DataFrame(survival_rows)

# Recommended use
def recommended_use(row):
    if row["mapping_status"] != "Mapped to clinical patient_id":
        return "CNV/TFx available but no clinical patient_id mapping found in the uploaded clinical file."

    if row["n_draws"] >= 5 and row["max_TFx"] >= 30 and row["max_CNV_burden"] >= 10:
        return "High-priority candidate for treatment-history + GRIFFIN/nucleosome profiling; multiple longitudinal draws with strong TFx/CNV signal."

    if row["n_draws"] >= 4 and row["max_TFx"] >= 20 and row["max_CNV_burden"] >= 8:
        return "Moderate-priority candidate; useful if treatment timing and GRIFFIN availability are good."

    return "Lower-priority candidate for integrated treatment/GRIFFIN analysis; review manually if biologically relevant."


candidate_cases_case_summary = (
    rank_features_all[
        [
            "Patient_Identifier", "patient_id", "Site", "mapping_status",
            "n_draws", "max_TFx", "min_TFx", "mean_TFx",
            "mean_CNV_burden", "max_CNV_burden",
            "CNV_rank_score", "notable_genes_states",
            "histology_from_ranked_or_annotation",
            "subtype_from_ranked_or_annotation"
        ]
    ]
    .merge(clinical_counts, on="Patient_Identifier", how="left")
    .merge(blood_days_summary, on="Patient_Identifier", how="left")
    .merge(receptor_summary, on="Patient_Identifier", how="left")
    .merge(metastatic_summary, on="Patient_Identifier", how="left")
    .merge(tx_summary, on="Patient_Identifier", how="left")
    .merge(survival_summary, on="Patient_Identifier", how="left")
)

candidate_cases_case_summary["recommended_use"] = candidate_cases_case_summary.apply(
    recommended_use,
    axis=1
)

candidate_cases_case_summary = candidate_cases_case_summary.rename(columns={
    "n_draws": "n_CNV_samples_in_matrix",
    "min_TFx": "TFx_percent_min",
    "max_TFx": "TFx_percent_max",
    "mean_TFx": "TFx_percent_mean"
})

candidate_cases_case_summary = candidate_cases_case_summary[
    [
        "Patient_Identifier", "patient_id", "Site", "mapping_status",
        "n_all_clinical_rows", "n_blood_draw_rows_in_clinical",
        "n_CNV_samples_in_matrix", "blood_draw_days",
        "TFx_percent_min", "TFx_percent_max", "TFx_percent_mean",
        "histology_from_ranked_or_annotation",
        "subtype_from_ranked_or_annotation",
        "CNV_rank_score",
        "mean_CNV_burden", "max_CNV_burden",
        "notable_genes_states",
        "receptor_summary_from_clinical",
        "metastatic_sites_summary",
        "treatment_summary",
        "survival_summary",
        "recommended_use"
    ]
].sort_values("CNV_rank_score", ascending=False)

# ============================================================
# 16. Save TSV files
# ============================================================

tables = {
    "all_cases_case_summary.tsv": candidate_cases_case_summary,
    "all_cases_blood_draws_with_treatment_CNV_TFx.tsv": candidate_cases_blood_draws_with_treatment_CNV_TFx,
    "all_cases_treatment_timeline.tsv": candidate_cases_treatment_timeline,
    "all_cases_diagnosis_receptor_tissue.tsv": candidate_cases_diagnosis_receptor_tissue,
    "all_cases_CNV_sample_summary.tsv": candidate_cases_CNV_sample_summary,
    "all_cases_CNV_gene_summary_key_genes.tsv": candidate_cases_CNV_gene_summary_key_genes,
    "all_cases_all_clinical_rows.tsv": candidate_cases_all_clinical_rows_export,
    "all_cases_CNV_rank_features.tsv": rank_features_all,
}

for fname, df in tables.items():
    df.to_csv(out_dir / fname, sep="\t", index=False)

print("Saved TSV files to:", out_dir)

# ============================================================
# 17. Save Excel workbook
# ============================================================

# For all-case analysis, do NOT write the full altered-gene string column
# into Excel because it can be very large.
candidate_cases_CNV_sample_summary_excel = candidate_cases_CNV_sample_summary.drop(
    columns=["Altered_genes_all_non_NEUT"],
    errors="ignore"
)

sheet_order = {
    "Case_summary": candidate_cases_case_summary,
    "Blood_draws": candidate_cases_blood_draws_with_treatment_CNV_TFx,
    "Treatment_timeline": candidate_cases_treatment_timeline,
    "Diagnosis_receptor": candidate_cases_diagnosis_receptor_tissue,
    "CNV_sample_summary": candidate_cases_CNV_sample_summary_excel,
    "CNV_gene_summary_key_genes": candidate_cases_CNV_gene_summary_key_genes,
    "All_clinical_rows": candidate_cases_all_clinical_rows_export,
    "CNV_rank_features_all": rank_features_all,
}

with pd.ExcelWriter(out_excel, engine="openpyxl") as writer:
    for sheet_name, df in sheet_order.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

        ws = writer.book[sheet_name]

        # Freeze header
        ws.freeze_panes = "A2"

        # Auto-filter
        if ws.max_row > 1 and ws.max_column > 1:
            ws.auto_filter.ref = ws.dimensions

        # Header style
        from openpyxl.styles import PatternFill, Font, Alignment

        header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
        header_font = Font(color="FFFFFF", bold=True)

        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        # Column widths
        for col_cells in ws.columns:
            col_letter = col_cells[0].column_letter
            header = str(col_cells[0].value)
            width = 22

            if any(k in header.lower() for k in [
                "summary", "genes", "treatment", "recommended",
                "states", "altered", "receptor", "metastatic", "clinical"
            ]):
                width = 40

            ws.column_dimensions[col_letter].width = width

        # Wrap text for all cells
        for row in ws.iter_rows():
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

print("Saved Excel workbook:", out_excel)

# ============================================================
# 18. Save ZIP of TSVs
# ============================================================

with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(out_dir.glob("*.tsv")):
        z.write(p, arcname=p.name)

print("Saved TSV ZIP:", out_zip)


# ============================================================
# 19. Final quick summary
# ============================================================

print("\n========== Final summary ==========")
print("Cases present in CNV matrix:", len(cnv_patients_all))
print("CNV samples:", len(cnv_sample_cols))
print("Mapped cases:", int((candidate_patient_map["mapping_status"] == "Mapped to clinical patient_id").sum()))
print("Unmapped/flagged cases:")
print(candidate_patient_map[candidate_patient_map["mapping_status"] != "Mapped to clinical patient_id"])

print("\nOutput Excel:")
print(out_excel)

print("\nOutput TSV ZIP:")
print(out_zip)


/ihome/alee/rak373/.local/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/ihome/alee/rak373/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Clinical rows: (6728, 33)
CNV matrix: (24881, 770)
CNV annotation: (769, 29)
Cases present in CNV matrix: 262
['AL1Y', 'AL4L', 'AL69', 'AL6N', 'AL9C', 'AL9W', 'AL9X', 'AL9Y', 'ALA8', 'ALA9', 'ALBN', 'ALD2', 'ALTD', 'ALTT', 'ALW9', 'AM01', 'AM2J', 'AM2M', 'AM9R', 'AMCZ', 'AME4', 'AMEA', 'AMFH', 'AMFP', 'AMHD', 'AMJN', 'AMJW', 'AMOD', 'AMOL', 'AMOP', 'AMRJ', 'AMT0', 'AMU9', 'AMUA', 'AMUY', 'AMUZ', 'AMVG', 'AMVP', 'AMVU', 'AMW2', 'AMWC', 'AMWE', 'AMZM', 'AN34', 'AN3U', 'AN3V', 'ANEY', 'ANF8', 'ANFC', 'ANFT', 'ANFU', 'ANFX', 'ANGM', 'ANMI', 'ANMM', 'ANN0', 'ANOR', 'ANPL', 'ANPW', 'ANQG', 'ANQU', 'ANQV', 'ANST', 'ANT6', 'ANTN', 'ANTO', 'ANTU', 'ANU1', 'ANWT', 'ANY5', 'ANYF', 'ANYM', 'ANYY', 'ANZA', 'AO01', 'AO02', 'AO0T', 'AO0U', 'AO17', 'AO18', 'AO1D', 'AO1E', 'AO1F', 'AO1Q', 'AO1T', 'AO1U', 'AO2J', 'AO2K', 'AO2P', 'AO2R', 'AO92', 'AO9I', 'AOAC', 'AOAZ', 'AOB4', 'AOC9', 'AODK', 'AOEH', 'AOET', 'AOEY', 'AOFA', 'AOFM', 'AOFY', 'AOGH', 'AOGX', 'AOHF', 'AOHG', 'AOHW', 'AOHX', 'AOI2', 'AOI3', '

/scratch/slurm-10298801/ipykernel_758189/2911381003.py:1320: UserWarning: Cell contents too long (46768), truncated to 32767 characters
  df.to_excel(writer, sheet_name=sheet_name, index=False)
/scratch/slurm-10298801/ipykernel_758189/2911381003.py:1320: UserWarning: Cell contents too long (42924), truncated to 32767 characters
  df.to_excel(writer, sheet_name=sheet_name, index=False)
/scratch/slurm-10298801/ipykernel_758189/2911381003.py:1320: UserWarning: Cell contents too long (34589), truncated to 32767 characters
  df.to_excel(writer, sheet_name=sheet_name, index=False)
/scratch/slurm-10298801/ipykernel_758189/2911381003.py:1320: UserWarning: Cell contents too long (32936), truncated to 32767 characters
  df.to_excel(writer, sheet_name=sheet_name, index=False)
/scratch/slurm-10298801/ipykernel_758189/2911381003.py:1320: UserWarning: Cell contents too long (43015), truncated to 32767 characters
  df.to_excel(writer, sheet_name=sheet_name, index=False)
/scratch/slurm-10298801/ipyker

Saved Excel workbook: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/All_cases_clinical_CNV_extraction_TSVs/All_cases_clinical_CNV_extraction.xlsx
Saved TSV ZIP: /ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/All_cases_clinical_CNV_extraction_TSVs/All_cases_clinical_CNV_extraction_TSVs.zip

========== Final summary ==========
Cases present in CNV matrix: 262
CNV samples: 769
Mapped cases: 262
Unmapped/flagged cases:
Empty DataFrame
Columns: [Patient_Identifier, patient_id, Site, mapping_status, patient_ids_all, n_clinical_barcode_rows, example_barcodes]
Index: []

Output Excel:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/All_cases_clinical_CNV_extraction_TSVs/All_cases_clinical_CNV_extraction.xlsx

Output TSV ZIP:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Clinical_data/All_cases_clinical_CNV_extraction_TSVs/All_cases_clinical_CNV_extraction_TSVs.zip
